# GPU-эксперимент: Dense@50 → multilingual reranker

Ноутбук сравнивает зафиксированный Dense top-50 с тремя cross-encoder reranker-моделями. Он **не пересчитывает embeddings и FAISS**, поэтому входной файл небольшой и все модели получают одинаковый набор кандидатов.

Перед запуском в Kaggle:
1. `Settings → Accelerator → GPU` (T4/P100 или лучше).
2. Включить Internet, чтобы скачать публичные модели с Hugging Face.
3. Добавить dataset с файлом `dense_mmr_unannotated.json`.

При наличии отдельно размеченного `textbook_qrels*.jsonl` ноутбук посчитает Hit/Recall/MAP/MRR/nDCG. Текущий шаблон qrels без `relevant_chunk_ids` метрик качества не даёт — это ожидаемо. Без qrels ноутбук всё равно сохранит reranked top-k и безопасные диагностические показатели, но не будет называть их Recall или accuracy. Длинные расчёты сохраняются каждые 1000 пар; cache применяется повторно только при совпадении fingerprint данных, модели и параметров, а CUDA OOM автоматически уменьшает batch size.

In [ ]:
%pip install -q -U "sentence-transformers>=5.0,<6" "transformers>=4.51,<5" accelerate pandas tqdm

In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import random
import shutil
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
CANDIDATE_K = 50       # один и тот же Dense pool для всех arms
SAVE_TOP_K = 50        # сохраняем весь pool для метрик @1/@5/@10/@50
BATCH_SIZE = 16        # стартовый batch; при CUDA OOM уменьшится автоматически
MAX_LENGTH = 768       # query + textbook chunk
CHECKPOINT_EVERY_PAIRS = 1_000
MAX_TASKS = None       # для smoke-test поставить 5, затем вернуть None
RESUME = True          # cache используется только при совпадении fingerprint

MODELS = {
    "gte_multilingual": {
        "name": "Alibaba-NLP/gte-multilingual-reranker-base",
        "revision": "8215cf0",
        "scorer": "cross_encoder",
        "batch_size": 16,
        "trust_remote_code": True,
    },
    "bge_v2_m3": {
        "name": "BAAI/bge-reranker-v2-m3",
        "revision": "953dc6f",
        "scorer": "cross_encoder",
        "batch_size": 16,
        "trust_remote_code": False,
    },
    "qwen3_reranker_06b": {
        "name": "Qwen/Qwen3-Reranker-0.6B",
        "revision": "e61197e",
        "scorer": "qwen_yes_no",
        "batch_size": 8,
        "trust_remote_code": False,
    },
}
RUN_MODELS = list(MODELS)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "GPU не найден: включите Accelerator = GPU в Kaggle"
DEVICE = "cuda"
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)

OUTPUT_DIR = Path("/kaggle/working/reranker_experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_DIR)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

def find_required_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(
            f"Не найден {name}. Добавьте Kaggle Dataset с этим файлом."
        )
    if len(matches) > 1:
        print(f"Найдено несколько {name}; используется {matches[0]}")
    return matches[0]

def find_optional_qrels() -> Path | None:
    patterns = ("textbook_qrels_llm.jsonl", "textbook_qrels.jsonl", "*qrels*.jsonl")
    for pattern in patterns:
        matches = sorted(INPUT_ROOT.rglob(pattern))
        if matches:
            return matches[0]
    return None

REPORT_PATH = find_required_file("dense_mmr_unannotated.json")
QRELS_PATH = find_optional_qrels()
REPORT_SHA256 = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
rows = report.get("per_query")
assert isinstance(rows, list) and rows, "В report отсутствует непустой per_query"
if MAX_TASKS is not None:
    rows = rows[:MAX_TASKS]

ids = [str(row.get("query_id") or "") for row in rows]
assert all(ids), "Каждая строка должна иметь query_id"
assert len(ids) == len(set(ids)), "query_id должны быть уникальными"

nonempty = sum(bool(row.get("dense", {}).get("candidates")) for row in rows)
pairs = sum(
    min(CANDIDATE_K, len(row.get("dense", {}).get("candidates", [])))
    for row in rows
)
print("Report:", REPORT_PATH)
print("Report SHA256:", REPORT_SHA256)
print(f"Tasks: {len(rows)}; rerankable: {nonempty}; empty Dense: {len(rows)-nonempty}")
print(f"Pairs per model: {pairs}")
print("Optional qrels:", QRELS_PATH or "не загружены")

In [ ]:
def dense_candidates(row: dict) -> list[dict]:
    candidates = list(row.get("dense", {}).get("candidates") or [])
    candidates.sort(key=lambda item: int(item.get("rank", 10**9)))
    return candidates[:CANDIDATE_K]

def query_text(row: dict) -> str:
    # Фиксируем тот же tool query: изменение query rewriting здесь не смешивается с reranker.
    return str(row.get("query") or "").strip()

def document_text(candidate: dict) -> str:
    # Metadata — часть reranker input, но не содержит reference answer.
    header = (
        f"subject: {candidate.get('subject') or 'unknown'}; "
        f"grade: {candidate.get('grade') or 'unknown'}; "
        f"textbook: {candidate.get('textbook') or 'unknown'}; "
        f"page: {candidate.get('page') or 'unknown'}"
    )
    return header + "\n" + str(candidate.get("text") or "")

def normalize_candidate(candidate: dict, *, new_rank: int, reranker_score=None) -> dict:
    payload = dict(candidate)
    payload["dense_rank"] = int(candidate.get("rank", new_rank))
    payload["dense_score"] = candidate.get("score")
    payload["rank"] = int(new_rank)
    if reranker_score is not None:
        payload["reranker_score"] = float(reranker_score)
    return payload

def write_rankings(path: Path, arm: str, rankings: dict[str, list[dict]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            query_id = str(row["query_id"])
            record = {
                "task_id": query_id,
                "query": row.get("query"),
                "subject": row.get("subject"),
                "retrieval_subject": row.get("retrieval_subject"),
                "grade": row.get("grade"),
                "arm": arm,
                "candidate_k": CANDIDATE_K,
                "top_k": SAVE_TOP_K,
                "rankings": rankings.get(query_id, []),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

def read_rankings(path: Path) -> dict[str, list[dict]]:
    result = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                result[str(record["task_id"])] = list(record.get("rankings") or [])
    return result

baseline = {
    str(row["query_id"]): [
        normalize_candidate(candidate, new_rank=rank)
        for rank, candidate in enumerate(dense_candidates(row)[:SAVE_TOP_K], 1)
    ]
    for row in rows
}
rankings_by_arm = {"dense": baseline}
write_rankings(OUTPUT_DIR / "rankings_dense.jsonl", "dense", baseline)

In [ ]:
QWEN_INSTRUCTION = (
    "Given a Turkish school question, retrieve textbook passages that provide "
    "the theory, formula, fact, or worked example needed to answer it"
)

def fingerprint_payload(payload: dict) -> str:
    encoded = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

def score_fingerprint(arm: str, config: dict, flat_refs: list[tuple[str, dict]]) -> str:
    return fingerprint_payload({
        "schema": "reranker-score-cache-v2",
        "report_sha256": REPORT_SHA256,
        "query_ids": [str(row["query_id"]) for row in rows],
        "candidate_ids": [str(candidate.get("chunk_id")) for _, candidate in flat_refs],
        "arm": arm,
        "config": config,
        "candidate_k": CANDIDATE_K,
        "max_length": MAX_LENGTH,
    })

def save_score_progress(
    path: Path, scores: np.ndarray, completed: int, batch_size: int,
    fingerprint: str, resolved_revision: str | None,
) -> None:
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(
        temporary,
        scores=scores.astype(np.float32),
        completed=np.asarray([completed], dtype=np.int64),
        batch_size=np.asarray([batch_size], dtype=np.int32),
        fingerprint=np.asarray(fingerprint),
        resolved_revision=np.asarray(resolved_revision or "unknown"),
    )
    temporary.replace(path)

def load_score_progress(
    path: Path, length: int, fingerprint: str, default_batch_size: int,
) -> tuple[np.ndarray, int, int, str | None]:
    empty = np.full(length, np.nan, dtype=np.float32)
    if not RESUME or not path.exists():
        return empty, 0, default_batch_size, None
    with np.load(path, allow_pickle=False) as cached:
        cached_fingerprint = str(cached["fingerprint"].item())
        cached_scores = cached["scores"].astype(np.float32)
        completed = int(cached["completed"][0])
        batch_size = int(cached["batch_size"][0])
        resolved_revision = str(cached["resolved_revision"].item())
    valid = (
        cached_fingerprint == fingerprint
        and len(cached_scores) == length
        and 0 <= completed <= length
        and np.isfinite(cached_scores[:completed]).all()
    )
    if not valid:
        print(f"Игнорируем несовместимый cache: {path.name}")
        return empty, 0, default_batch_size, None
    print(f"Продолжаем {path.stem}: {completed}/{length}, batch_size={batch_size}")
    return cached_scores, completed, batch_size, resolved_revision

def validate_arm_rankings(arm: str, rankings: dict[str, list[dict]]) -> None:
    expected_query_ids = [str(row["query_id"]) for row in rows]
    if set(rankings) != set(expected_query_ids):
        raise ValueError(f"{arm}: набор query_id не совпал")
    for row in rows:
        query_id = str(row["query_id"])
        known = dense_candidates(row)
        known_ids = {str(item.get("chunk_id")) for item in known}
        items = rankings[query_id]
        expected_length = min(SAVE_TOP_K, len(known))
        if len(items) != expected_length:
            raise ValueError(f"{arm}/{query_id}: {len(items)} ranks вместо {expected_length}")
        ids = [str(item.get("chunk_id")) for item in items]
        if len(ids) != len(set(ids)):
            raise ValueError(f"{arm}/{query_id}: повтор chunk_id")
        if not set(ids).issubset(known_ids):
            raise ValueError(f"{arm}/{query_id}: неизвестный chunk_id")
        if SAVE_TOP_K >= len(known) and set(ids) != known_ids:
            raise ValueError(f"{arm}/{query_id}: изменился candidate set")
        if [item.get("rank") for item in items] != list(range(1, len(items) + 1)):
            raise ValueError(f"{arm}/{query_id}: некорректные rank")
        if arm != "dense" and not all(
            math.isfinite(float(item.get("reranker_score"))) for item in items
        ):
            raise ValueError(f"{arm}/{query_id}: NaN/Inf reranker score")

def cross_encoder_scores(
    model: CrossEncoder, pairs: list[tuple[str, str]], batch_size: int,
) -> np.ndarray:
    values = model.predict(
        pairs, batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True
    )
    return np.asarray(values, dtype=np.float32).reshape(-1)

def load_qwen_scorer(config: dict) -> dict:
    tokenizer = AutoTokenizer.from_pretrained(
        config["name"], revision=config.get("revision"), padding_side="left"
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        config["name"], revision=config.get("revision"),
        torch_dtype=torch.float16, attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    ).to(DEVICE).eval()
    no_ids = tokenizer("no", add_special_tokens=False).input_ids
    yes_ids = tokenizer("yes", add_special_tokens=False).input_ids
    if len(no_ids) != 1 or len(yes_ids) != 1:
        raise ValueError(f"Qwen yes/no должны быть однотокенными: {yes_ids=} {no_ids=}")
    prefix = (
        '<|im_start|>system\nJudge whether the Document meets the requirements '
        'based on the Query and the Instruct provided. Note that the answer can '
        'only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n'
    )
    suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
    prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)
    available_length = MAX_LENGTH - len(prefix_tokens) - len(suffix_tokens)
    if available_length < 128:
        raise ValueError("Слишком мало токенов осталось для Qwen query+document")
    return {
        "model": model, "tokenizer": tokenizer,
        "no_token_id": int(no_ids[0]), "yes_token_id": int(yes_ids[0]),
        "prefix_tokens": prefix_tokens, "suffix_tokens": suffix_tokens,
        "available_length": available_length,
    }

def qwen_scores(resources: dict, pairs: list[tuple[str, str]], batch_size: int) -> np.ndarray:
    model = resources["model"]
    tokenizer = resources["tokenizer"]
    output = np.empty(len(pairs), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(pairs), batch_size):
            stop = min(start + batch_size, len(pairs))
            formatted = [
                f"<Instruct>: {QWEN_INSTRUCTION}\n<Query>: {query}\n<Document>: {document}"
                for query, document in pairs[start:stop]
            ]
            encoded = tokenizer(
                formatted, padding=False, truncation=True,
                max_length=resources["available_length"], add_special_tokens=False,
            )
            encoded["input_ids"] = [
                resources["prefix_tokens"] + ids + resources["suffix_tokens"]
                for ids in encoded["input_ids"]
            ]
            tensors = tokenizer.pad(
                {"input_ids": encoded["input_ids"]}, padding=True,
                return_attention_mask=True, return_tensors="pt",
            )
            tensors = {key: value.to(DEVICE) for key, value in tensors.items()}
            logits = model(
                **tensors, use_cache=False, logits_to_keep=1, return_dict=True
            ).logits[:, -1, :]
            yes_no = torch.stack([
                logits[:, resources["no_token_id"]],
                logits[:, resources["yes_token_id"]],
            ], dim=1)
            output[start:stop] = torch.softmax(yes_no.float(), dim=1)[:, 1].cpu().numpy()
    return output

def run_reranker(arm: str, config: dict) -> tuple[dict[str, list[dict]], dict]:
    output_path = OUTPUT_DIR / f"rankings_{arm}.jsonl"
    score_cache_path = OUTPUT_DIR / f"score_cache_{arm}.npz"
    flat_pairs: list[tuple[str, str]] = []
    flat_refs: list[tuple[str, dict]] = []
    for row in rows:
        query_id = str(row["query_id"])
        query = query_text(row)
        for candidate in dense_candidates(row):
            flat_pairs.append((query, document_text(candidate)))
            flat_refs.append((query_id, candidate))

    fingerprint = score_fingerprint(arm, config, flat_refs)
    default_batch_size = int(config.get("batch_size", BATCH_SIZE))
    scores, start, effective_batch_size, cached_revision = load_score_progress(
        score_cache_path, len(flat_pairs), fingerprint, default_batch_size
    )
    resumed_from = start
    model = None
    resources = None
    load_seconds = 0.0
    resolved_revision = cached_revision
    if start < len(flat_pairs):
        load_started = time.perf_counter()
        if config["scorer"] == "cross_encoder":
            model = CrossEncoder(
                config["name"], device=DEVICE, max_length=MAX_LENGTH,
                trust_remote_code=bool(config.get("trust_remote_code", False)),
                revision=config.get("revision"),
                model_kwargs={"torch_dtype": torch.float16, "low_cpu_mem_usage": True},
            )
            resolved_revision = str(
                getattr(getattr(model.model, "config", None), "_commit_hash", None)
                or config.get("revision") or "unknown"
            )
        elif config["scorer"] == "qwen_yes_no":
            resources = load_qwen_scorer(config)
            resolved_revision = str(
                getattr(resources["model"].config, "_commit_hash", None)
                or config.get("revision") or "unknown"
            )
        else:
            raise ValueError(f"Неизвестный scorer: {config['scorer']}")
        load_seconds = time.perf_counter() - load_started

    inference_started = time.perf_counter()
    while start < len(flat_pairs):
        stop = min(start + CHECKPOINT_EVERY_PAIRS, len(flat_pairs))
        segment = flat_pairs[start:stop]
        while True:
            try:
                if config["scorer"] == "cross_encoder":
                    segment_scores = cross_encoder_scores(model, segment, effective_batch_size)
                else:
                    segment_scores = qwen_scores(resources, segment, effective_batch_size)
                break
            except torch.cuda.OutOfMemoryError:
                if effective_batch_size <= 1:
                    raise
                effective_batch_size = max(1, effective_batch_size // 2)
                print(f"[{arm}] CUDA OOM; новый batch_size={effective_batch_size}")
                gc.collect()
                torch.cuda.empty_cache()
        if len(segment_scores) != len(segment) or not np.isfinite(segment_scores).all():
            raise ValueError(f"{arm}: некорректные scores в диапазоне {start}:{stop}")
        scores[start:stop] = segment_scores
        start = stop
        save_score_progress(
            score_cache_path, scores, start, effective_batch_size,
            fingerprint, resolved_revision,
        )
        print(f"[{arm}] checkpoint {start}/{len(flat_pairs)}")
    inference_seconds = time.perf_counter() - inference_started
    if not np.isfinite(scores).all():
        raise ValueError(f"{arm}: итоговый cache содержит NaN/Inf")

    grouped: dict[str, list[tuple[dict, float]]] = defaultdict(list)
    for (query_id, candidate), score in zip(flat_refs, scores):
        grouped[query_id].append((candidate, float(score)))
    reranked: dict[str, list[dict]] = {}
    for row in rows:
        query_id = str(row["query_id"])
        ordered = sorted(
            grouped.get(query_id, []),
            key=lambda item: (
                -item[1], int(item[0].get("rank", 10**9)), str(item[0].get("chunk_id")),
            ),
        )
        reranked[query_id] = [
            normalize_candidate(candidate, new_rank=rank, reranker_score=score)
            for rank, (candidate, score) in enumerate(ordered[:SAVE_TOP_K], 1)
        ]

    validate_arm_rankings(arm, reranked)
    write_rankings(output_path, arm, reranked)
    roundtrip = read_rankings(output_path)
    validate_arm_rankings(arm, roundtrip)
    processed_pairs = len(flat_pairs) - resumed_from
    stats = {
        "arm": arm, "model": config["name"], "scorer": config["scorer"],
        "requested_revision": config.get("revision"),
        "resolved_revision": resolved_revision,
        "pairs": len(flat_pairs), "resumed_from_pairs": resumed_from,
        "load_seconds": round(load_seconds, 3),
        "inference_seconds": round(inference_seconds, 3),
        "pairs_per_second": (
            round(processed_pairs / inference_seconds, 3) if inference_seconds > 0 else None
        ),
        "effective_batch_size": effective_batch_size,
        "score_fingerprint": fingerprint,
    }
    del model, resources, scores
    gc.collect()
    torch.cuda.empty_cache()
    return roundtrip, stats

validate_arm_rankings("dense", baseline)
validate_arm_rankings("dense", read_rankings(OUTPUT_DIR / "rankings_dense.jsonl"))
runtime_rows = []
for arm in RUN_MODELS:
    print(f"\n=== {arm}: {MODELS[arm]['name']} ===")
    arm_rankings, stats = run_reranker(arm, MODELS[arm])
    rankings_by_arm[arm] = arm_rankings
    runtime_rows.append(stats)

runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(OUTPUT_DIR / "runtime.csv", index=False)
runtime_df

In [ ]:
def candidate_ids(items: list[dict], k: int) -> list[str]:
    return [str(item.get("chunk_id")) for item in items[:k] if item.get("chunk_id")]

def unique_pages(items: list[dict], k: int = 5) -> int:
    pages = {
        (item.get("textbook"), item.get("page"))
        for item in items[:k]
        if item.get("textbook") is not None or item.get("page") is not None
    }
    return len(pages)

diagnostic_rows = []
for arm, arm_rankings in rankings_by_arm.items():
    if arm == "dense":
        continue
    comparable = 0
    top1_changed = 0
    top5_set_changed = 0
    page_counts = []
    for query_id, dense_items in baseline.items():
        reranked_items = arm_rankings.get(query_id, [])
        if not dense_items or not reranked_items:
            continue
        comparable += 1
        top1_changed += candidate_ids(dense_items, 1) != candidate_ids(reranked_items, 1)
        top5_set_changed += set(candidate_ids(dense_items, 5)) != set(candidate_ids(reranked_items, 5))
        page_counts.append(unique_pages(reranked_items, 5))
    diagnostic_rows.append({
        "arm": arm,
        "comparable_tasks": comparable,
        "top1_changed_rate": top1_changed / comparable if comparable else None,
        "top5_set_changed_rate": top5_set_changed / comparable if comparable else None,
        "mean_unique_pages_at_5": float(np.mean(page_counts)) if page_counts else None,
    })

diagnostics_df = pd.DataFrame(diagnostic_rows)
diagnostics_df.to_csv(OUTPUT_DIR / "diagnostics.csv", index=False)
print("Эти показатели описывают изменение выдачи, а не её правильность.")
diagnostics_df

In [ ]:
CUTOFFS = (1, 5, 10, 50)

def load_qrels(path: Path | None) -> dict[str, set[str]]:
    if path is None:
        return {}
    result = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            query_id = str(row.get("query_id") or row.get("task_id") or "")
            relevant = {str(value) for value in row.get("relevant_chunk_ids", []) if value}
            if query_id and relevant:
                result[query_id] = relevant
    return result

def ranking_metrics(ranked_ids: list[str], relevant: set[str], k: int) -> dict:
    top = ranked_ids[:k]
    hit_positions = [index for index, chunk_id in enumerate(top, 1) if chunk_id in relevant]
    hit_count = len(hit_positions)
    precision_sum = 0.0
    seen_relevant = 0
    dcg = 0.0
    for index, chunk_id in enumerate(top, 1):
        if chunk_id in relevant:
            seen_relevant += 1
            precision_sum += seen_relevant / index
            dcg += 1.0 / math.log2(index + 1)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(index + 1) for index in range(1, ideal_hits + 1))
    return {
        f"hit@{k}": float(bool(hit_positions)),
        f"recall@{k}": hit_count / len(relevant),
        f"map@{k}": precision_sum / ideal_hits,
        f"mrr@{k}": 1.0 / hit_positions[0] if hit_positions else 0.0,
        f"ndcg@{k}": dcg / idcg if idcg else 0.0,
    }

qrels = load_qrels(QRELS_PATH)
row_by_id = {str(row["query_id"]): row for row in rows}
metric_rows = []
for arm, arm_rankings in rankings_by_arm.items():
    for query_id, relevant in qrels.items():
        if query_id not in row_by_id:
            continue
        ranked_ids = candidate_ids(arm_rankings.get(query_id, []), SAVE_TOP_K)
        record = {
            "arm": arm,
            "query_id": query_id,
            "subject": row_by_id[query_id].get("subject") or "unknown",
        }
        for cutoff in CUTOFFS:
            record.update(ranking_metrics(ranked_ids, relevant, cutoff))
        metric_rows.append(record)

if not metric_rows:
    print(
        "Нет задач с непустыми relevant_chunk_ids. "
        "Rankings сохранены, но Recall/MAP/MRR/nDCG намеренно не рассчитаны."
    )
    metrics_overall = pd.DataFrame()
    metrics_by_subject = pd.DataFrame()
else:
    per_query_metrics = pd.DataFrame(metric_rows)
    metric_columns = [column for column in per_query_metrics.columns if "@" in column]
    metrics_overall = per_query_metrics.groupby("arm")[metric_columns].mean().reset_index()
    metrics_by_subject = (
        per_query_metrics.groupby(["arm", "subject"])[metric_columns]
        .mean()
        .reset_index()
    )
    per_query_metrics.to_csv(OUTPUT_DIR / "metrics_per_query.csv", index=False)
    metrics_overall.to_csv(OUTPUT_DIR / "metrics_overall.csv", index=False)
    metrics_by_subject.to_csv(OUTPUT_DIR / "metrics_by_subject.csv", index=False)

metrics_overall

In [ ]:
artifact_sha256 = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(OUTPUT_DIR.glob("rankings_*.jsonl"))
}
manifest = {
    "schema_version": "kaggle-reranker-experiment-v2",
    "input_report": REPORT_PATH.name,
    "input_report_sha256": REPORT_SHA256,
    "qrels": QRELS_PATH.name if QRELS_PATH else None,
    "annotated_qrels": len(qrels),
    "tasks": len(rows),
    "rerankable_tasks": nonempty,
    "candidate_k": CANDIDATE_K,
    "save_top_k": SAVE_TOP_K,
    "max_length": MAX_LENGTH,
    "checkpoint_every_pairs": CHECKPOINT_EVERY_PAIRS,
    "models": {arm: MODELS[arm] for arm in RUN_MODELS},
    "runtime": runtime_rows,
    "determinism": {
        "seed": SEED, "tf32": False,
        "torch_deterministic_warn_only": True,
        "tie_break": "reranker_score desc, dense_rank asc, chunk_id asc",
    },
    "validation": {
        "query_ids_unique": True,
        "candidate_sets_preserved": SAVE_TOP_K >= CANDIDATE_K,
        "ranks_contiguous": True,
        "scores_finite": True,
        "jsonl_roundtrip_checked": True,
    },
    "artifact_sha256": artifact_sha256,
    "device": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "transformers_version": importlib_metadata.version("transformers"),
    "sentence_transformers_version": importlib_metadata.version("sentence-transformers"),
}
(OUTPUT_DIR / "experiment_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
archive = shutil.make_archive(
    "/kaggle/working/reranker_experiment_outputs",
    "zip",
    OUTPUT_DIR,
)
archive_sha256 = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
Path(archive + ".sha256").write_text(archive_sha256 + "  " + Path(archive).name + "\n")
print("Готово:", archive)
print("SHA256:", archive_sha256)
print("Скачайте ZIP через правую панель Kaggle → Output.")
sorted(path.name for path in OUTPUT_DIR.iterdir())

## Как интерпретировать результат

- Если есть qrels, основной выбор делайте по `nDCG@5`, `MRR@10` и `Recall@5`, затем проверяйте latency.
- `Recall@50` у всех reranker arms обязан совпадать с Dense, потому что reranker только переставляет фиксированные 50 кандидатов.
- `top1_changed_rate` и `top5_set_changed_rate` не являются показателями качества — они лишь подтверждают, что модель действительно изменила порядок.
- После выбора победителя его top-5 нужно подать в один и тот же agent+judge pipeline и сравнить strict answer accuracy, fixed/regressed и McNemar с Dense baseline.